#Dúvidas a se confirmar

0. COX-2 como exemplo, ok?

1. Ativa ≤ 1 µM e inativa ≥ 10 µM, ok?
O ChEMBL dá só um número (o IC50), e o SVM precisa de sim/não. Esses são os limites escolhidos para separar as classes, e o que fica no meio é descartado.

2. Ajustar C e gamma ou deixar o padrão?
estão no valor padrão. precisa de ajustar esses parametros?

3. Divisão ao acaso ou por scaffold?
Ao acaso, moléculas quase iguais caem no treino e no teste, e o resultado fica otimista. Por scaffold, o teste só tem famílias novas: é mais realista, mas a nota cai (0,84 → 0,76).

4. Métricas: acurácia balanceada, MCC e ROC-AUC, ok?



# SVM Activity Classifier

## Objective and overview

This notebook trains a binary support vector machine (SVM) with a radial basis
function (RBF) kernel to classify molecules as active or inactive, using Morgan
fingerprint bit-vectors computed with RDKit.

The SVM finds the boundary that separates the two classes with the largest
margin; the RBF kernel allows this boundary to be non-linear. The trained model
predicts the class of new molecules and returns a score that can be used to
rank them.

## Inputs and setup

The training data must be a CSV file with:

- `smiles`: the molecular structure as a SMILES string.
- `activity`: the known class (`active` or `inactive`).

Labels must come from a scientifically justified criterion for the target and
assay studied. Dependencies: RDKit, scikit-learn, pandas, NumPy and, for the
worked example only, `chembl_webresource_client`.

In [1]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                             confusion_matrix, matthews_corrcoef, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

RDLogger.DisableLog("rdApp.*")  # invalid SMILES are reported by the functions below

All settings are defined here. Values marked as provisional are choices for this
example and can be changed for other studies.

In [2]:
DATA_FILE = "data/chembl_cox2_ic50.csv"
PREDICTIONS_FILE = "svm_predictions.csv"

SMILES_COLUMN = "smiles"
TARGET_COLUMN = "activity"
POSITIVE_LABEL = "active"
NEGATIVE_LABEL = "inactive"

MORGAN_RADIUS = 2               # provisional (ECFP4)
MORGAN_SIZE = 2048              # provisional

SVM_C = 1.0                     # provisional (scikit-learn default)
SVM_GAMMA = "scale"             # provisional (scikit-learn default)
SVM_CLASS_WEIGHT = "balanced"   # provisional, compensates unbalanced classes

TEST_SIZE = 0.20
RANDOM_STATE = 42

## Data preparation (worked example)

The example uses IC50 values for human cyclooxygenase-2 (COX-2, ChEMBL target
`CHEMBL230`) from ChEMBL
(https://www.ebi.ac.uk/chembl/explore/target/CHEMBL230). COX-2 was chosen as a well-known target with many
measurements.

Selection criteria:

- `standard_type = IC50`, `standard_units = nM`, relation `=`, `>` or `>=`.
- Censored values (`>`, `>=`) are kept only when they are at or above the
  inactive threshold, because they then prove inactivity.
- Repeated measurements are summarized by the median per molecule.
- IC50 ≤ 1 µM → `active`; IC50 ≥ 10 µM → `inactive`; intermediate values are
  discarded. These thresholds are a choice for this example, not a general rule.

The CSV in `data/` was downloaded on 2026-10-14 (ChEMBL 37). The next cell only
downloads the data again if the file is missing. Any CSV with `smiles` and
`activity` columns can replace this step.

In [3]:
TARGET_CHEMBL_ID = "CHEMBL230"   # COX-2, human
ACTIVE_THRESHOLD_NM = 1_000      # provisional
INACTIVE_THRESHOLD_NM = 10_000   # provisional


def fetch_chembl_ic50(target_chembl_id):
    """Download IC50 measurements (nM) for one ChEMBL target."""
    from chembl_webresource_client.new_client import new_client

    records = new_client.activity.filter(
        target_chembl_id=target_chembl_id,
        standard_type="IC50",
        standard_relation__in=["=", ">", ">="],
        standard_units="nM",
    ).only(["molecule_chembl_id", "canonical_smiles", "standard_relation", "standard_value"])

    data = pd.DataFrame(list(records))
    if data.empty:
        raise ValueError(f"No IC50 data found for target {target_chembl_id}.")

    # The API also returns unrequested fields (e.g. "relation"), so keep only the needed ones.
    data = data[["molecule_chembl_id", "canonical_smiles", "standard_relation", "standard_value"]]
    data = data.rename(columns={"canonical_smiles": SMILES_COLUMN,
                                "standard_relation": "relation",
                                "standard_value": "ic50_nm"})
    data["ic50_nm"] = pd.to_numeric(data["ic50_nm"], errors="coerce")
    return data


def label_by_potency(activity_data, active_threshold_nm, inactive_threshold_nm):
    """Aggregate IC50 values per molecule and assign active/inactive labels."""
    if active_threshold_nm >= inactive_threshold_nm:
        raise ValueError("The active threshold must be lower than the inactive threshold.")

    data = activity_data.dropna(subset=[SMILES_COLUMN, "ic50_nm"])
    censored = data["relation"] != "="
    data = data[~censored | (data["ic50_nm"] >= inactive_threshold_nm)]

    molecules = (data.groupby("molecule_chembl_id", as_index=False)
                 .agg(smiles=(SMILES_COLUMN, "first"), ic50_nm=("ic50_nm", "median")))
    molecules[TARGET_COLUMN] = pd.NA
    molecules.loc[molecules["ic50_nm"] <= active_threshold_nm, TARGET_COLUMN] = POSITIVE_LABEL
    molecules.loc[molecules["ic50_nm"] >= inactive_threshold_nm, TARGET_COLUMN] = NEGATIVE_LABEL

    labeled = molecules.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
    if labeled[TARGET_COLUMN].nunique() < 2:
        raise ValueError("Both active and inactive molecules are required after labeling.")
    return labeled

In [4]:
if not Path(DATA_FILE).exists():
    labeled = label_by_potency(fetch_chembl_ic50(TARGET_CHEMBL_ID),
                               ACTIVE_THRESHOLD_NM, INACTIVE_THRESHOLD_NM)
    Path(DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
    labeled.to_csv(DATA_FILE, index=False)
    print(f"Downloaded on {date.today()} and saved to {DATA_FILE}")

## Implementation with a worked example

### Loading the data

`load_dataset` checks the required columns and removes repeated structures
(same canonical SMILES). Structures with conflicting labels are discarded.

In [5]:
def load_dataset(path, smiles_column=SMILES_COLUMN, target_column=TARGET_COLUMN):
    """Load a labeled CSV and remove invalid and repeated structures."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}. Run the data preparation step "
                                "or provide your own CSV.")
    data = pd.read_csv(path)
    missing = {smiles_column, target_column} - set(data.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {', '.join(sorted(missing))}.")

    data = data.dropna(subset=[smiles_column, target_column]).copy()
    molecules = [Chem.MolFromSmiles(str(smiles)) for smiles in data[smiles_column]]
    valid = [molecule is not None for molecule in molecules]
    if not any(valid):
        raise ValueError("None of the SMILES strings could be parsed.")
    data["canonical_smiles"] = [Chem.MolToSmiles(m) if m is not None else None for m in molecules]
    data = data[valid]

    n_labels = data.groupby("canonical_smiles")[target_column].transform("nunique")
    data = data[n_labels == 1].drop_duplicates("canonical_smiles")

    print(f"Loaded {len(data)} molecules ({sum(not v for v in valid)} invalid SMILES removed).")
    return data.reset_index(drop=True)


dataset = load_dataset(DATA_FILE)
dataset[TARGET_COLUMN].value_counts()

Loaded 4105 molecules (0 invalid SMILES removed).


activity
active      2427
inactive    1678
Name: count, dtype: int64

### Morgan fingerprints

Each molecule becomes a bit-vector: each bit marks the presence of a circular
substructure (an atom and its neighbors up to `MORGAN_RADIUS` bonds away).

In [6]:
def morgan_bit_vectors(smiles_list, radius=MORGAN_RADIUS, size=MORGAN_SIZE):
    """Convert SMILES strings into a matrix of Morgan bit-vectors."""
    if radius < 0 or size <= 0:
        raise ValueError("Radius must be >= 0 and size must be > 0.")
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=size)

    fingerprints = []
    for row, smiles in enumerate(smiles_list):
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is None:
            raise ValueError(f"Invalid SMILES at position {row}: {smiles}")
        fingerprints.append(generator.GetFingerprintAsNumPy(molecule))
    return np.array(fingerprints, dtype=np.uint8)


def encode_labels(labels, positive_label=POSITIVE_LABEL):
    """Convert class labels into 1 (positive) and 0 (negative)."""
    labels = pd.Series(labels).astype(str)
    classes = set(labels.unique())
    if positive_label not in classes or len(classes) != 2:
        raise ValueError(f"Expected exactly two classes including '{positive_label}', "
                         f"found: {sorted(classes)}.")
    return (labels == positive_label).astype(int).to_numpy()


features = morgan_bit_vectors(dataset[SMILES_COLUMN])
labels = encode_labels(dataset[TARGET_COLUMN])
print("Feature matrix:", features.shape)

Feature matrix: (4105, 2048)


### Train/test split

A stratified split keeps the same proportion of actives and inactives in the
training and test sets.

In [7]:
train_idx, test_idx = train_test_split(np.arange(len(dataset)), test_size=TEST_SIZE,
                                       stratify=labels, random_state=RANDOM_STATE)
print(f"Training: {len(train_idx)} | Test: {len(test_idx)}")

Training: 3284 | Test: 821


### Training the SVM

`C` controls how strongly training errors are penalized and `gamma` how
flexible the RBF boundary is. `class_weight="balanced"` gives more weight to the
rarer class.

In [8]:
def train_svm(features, labels, c=SVM_C, gamma=SVM_GAMMA, class_weight=SVM_CLASS_WEIGHT):
    """Train a binary SVM classifier with an RBF kernel."""
    model = SVC(kernel="rbf", C=c, gamma=gamma, class_weight=class_weight)
    return model.fit(features, labels)


model = train_svm(features[train_idx], labels[train_idx])
print("Support vectors per class (inactive, active):", model.n_support_)

Support vectors per class (inactive, active): [1008 1317]


### Evaluation

Accuracy alone is misleading when classes are unbalanced, so balanced accuracy,
the Matthews correlation coefficient (MCC) and ROC-AUC are also reported.
ROC-AUC uses the SVM score and measures how well actives are ranked above
inactives.

In [9]:
def evaluate_classifier(model, features, labels):
    """Print classification metrics suited to unbalanced data."""
    predicted = model.predict(features)
    scores = model.decision_function(features)
    print(classification_report(labels, predicted, labels=[0, 1],
                                target_names=[NEGATIVE_LABEL, POSITIVE_LABEL], digits=3))
    confusion = pd.DataFrame(confusion_matrix(labels, predicted, labels=[0, 1]),
                             index=pd.Index([NEGATIVE_LABEL, POSITIVE_LABEL], name="Actual"),
                             columns=pd.Index([NEGATIVE_LABEL, POSITIVE_LABEL], name="Predicted"))
    print(confusion, "\n")
    print(f"Balanced accuracy: {balanced_accuracy_score(labels, predicted):.3f}")
    print(f"MCC:               {matthews_corrcoef(labels, predicted):.3f}")
    print(f"ROC-AUC:           {roc_auc_score(labels, scores):.3f}")


evaluate_classifier(model, features[test_idx], labels[test_idx])

              precision    recall  f1-score   support

    inactive      0.802     0.830     0.816       336
      active      0.879     0.858     0.868       485

    accuracy                          0.847       821
   macro avg      0.841     0.844     0.842       821
weighted avg      0.848     0.847     0.847       821

Predicted  inactive  active
Actual                     
inactive        279      57
active           69     416 

Balanced accuracy: 0.844
MCC:               0.685
ROC-AUC:           0.922


### Ranking new molecules

`rank_molecules` predicts the class of any table with a `smiles` column and
sorts it by SVM score (higher = more likely active). Here the held-out test
molecules play the role of new candidates; for a real screen, load your own CSV.

In [10]:
def rank_molecules(model, molecules, smiles_column=SMILES_COLUMN):
    """Predict the class of new molecules and rank them by SVM score."""
    if smiles_column not in molecules.columns:
        raise ValueError(f"Missing column: {smiles_column}")
    features = morgan_bit_vectors(molecules[smiles_column])
    ranked = molecules.copy()
    ranked["svm_score"] = model.decision_function(features)
    ranked["predicted_activity"] = np.where(model.predict(features) == 1,
                                            POSITIVE_LABEL, NEGATIVE_LABEL)
    return ranked.sort_values("svm_score", ascending=False).reset_index(drop=True)


candidates = dataset.iloc[test_idx].drop(columns=["canonical_smiles"])
ranking = rank_molecules(model, candidates)
ranking.to_csv(PREDICTIONS_FILE, index=False)
ranking.head(10)

,molecule_chembl_id,smiles,ic50_nm,activity,svm_score,predicted_activity
0,CHEMBL29403,Cc1cc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)...,52.240,active,1.571399,active
1,CHEMBL312822,NS(=O)(=O)c1ccc(C2=C(c3ccc(F)c(Cl)c3)CC3(CC3)C...,1.507,active,1.529656,active
2,CHEMBL4466770,N#Cc1cnn(-c2ccc(S(N)(=O)=O)cc2)c1/N=C/c1ccc(Cl...,189.770,active,1.465578,active
3,CHEMBL88550,COc1ccc(C2=C(c3ccc(S(C)(=O)=O)cc3)CC3(CC3)C2)cc1F,10.000,active,1.451562,active
4,CHEMBL5282037,Cc1nnc(/N=C2\NC(=O)/C(=C/c3ccc(Cl)cc3)S2)s1,100.000,active,1.435233,active
5,CHEMBL374427,COC1(c2cc(F)cc(OCc3cc(-c4ccccc4)n(-c4ccc(S(N)(...,100.000,active,1.422061,active
6,CHEMBL322807,CC1(C)OC(=O)C(Oc2ncccc2Cl)=C1c1ccc(S(C)(=O)=O)cc1,90.000,active,1.408465,active
7,CHEMBL4567195,N#Cc1cnn(-c2ccc(S(N)(=O)=O)cc2)c1/N=C/c1ccc(Br...,64.900,active,1.400816,active
8,CHEMBL5285552,Cc1nnc(/N=C2\NC(=O)/C(=C/c3ccc(Br)cc3)S2)s1,100.000,active,1.370158,active
9,CHEMBL296662,COc1ccc(-c2nc(C(F)(F)F)cn2-c2ccc(S(N)(=O)=O)cc...,30.100,active,1.358445,active


## Results, limitations and references

In the worked example (COX-2, 4,105 molecules, 821 in the test set), the SVM
reached a balanced accuracy of 0.84, MCC of 0.69 and ROC-AUC of 0.92. The top of
the ranking is dominated by known actives, which is the intended use in virtual
screening: prioritizing molecules for experimental testing.

Limitations:

- A random split places close analogs in both training and test sets, so the
  metrics are likely optimistic for new chemical scaffolds; a scaffold-based
  split gives a stricter estimate.
- `C` and `gamma` were not tuned; tuning should use cross-validation on the
  training set only, never the test set.
- Labels depend on the chosen thresholds and mix IC50 values from different
  assays and laboratories.
- Predictions are only reliable for molecules similar to the training data
  and require experimental confirmation.

References:

1. Oliveira, T.A.; Silva, M.P.; Maia, E.H.B.; Silva, A.M.; Taranto, A.G. Virtual
   Screening Algorithms in Drug Discovery: A Review Focused on Machine and Deep
   Learning Methods. *Drugs Drug Candidates* **2023**, 2, 311–334.
   https://doi.org/10.3390/ddc2020017
2. Cortes, C.; Vapnik, V. Support-vector networks. *Mach. Learn.* **1995**, 20,
   273–297. https://doi.org/10.1007/BF00994018
3. Rogers, D.; Hahn, M. Extended-connectivity fingerprints. *J. Chem. Inf. Model.*
   **2010**, 50, 742–754. https://doi.org/10.1021/ci100050t
4. Zdrazil, B. et al. The ChEMBL Database in 2023: a drug discovery platform
   spanning multiple bioactivity data types and time periods. *Nucleic Acids Res.* **2024**,
   52, D1180–D1192. https://doi.org/10.1093/nar/gkad1004
5. Pedregosa, F. et al. Scikit-learn: Machine Learning in Python. *J. Mach. Learn.
   Res.* **2011**, 12, 2825–2830.
6. RDKit: Open-source cheminformatics. https://www.rdkit.org